In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "utils" / "paths.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook inside the project directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.paths import RAW, stage_path
from utils.runs import require_stage, stage_run
from loguru import logger
import csv
from collections import Counter
from pathlib import Path

from loguru import logger
from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, QED
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

In [ ]:
input_filename = require_stage("02_lipinski") / "lipinski_passed.smi"
CALCULATE_STRUCTURAL_ALERTS = True

In [ ]:
# Turn this off for very large libraries if alert matching takes too long.
CALCULATE_STRUCTURAL_ALERTS = True


def build_catalog(kind):
    params = FilterCatalogParams()
    params.AddCatalog(kind)
    return FilterCatalog(params)


if CALCULATE_STRUCTURAL_ALERTS:
    pains_catalog = build_catalog(FilterCatalogParams.FilterCatalogs.PAINS)
    brenk_catalog = build_catalog(FilterCatalogParams.FilterCatalogs.BRENK)
    logger.info('PAINS/Brenk catalogs loaded (informational alerts only).')
else:
    pains_catalog = None
    brenk_catalog = None

In [ ]:
PROPERTY_COLUMNS = [
    'MW', 'ExactMW', 'TPSA', 'cLogP', 'MolMR', 'QED',
    'RotatableBonds', 'HBD', 'HBA', 'FormalCharge',
    'HeavyAtomCount', 'HeteroatomCount',
    'RingCount', 'AromaticRings', 'AliphaticRings',
    'SaturatedRings', 'Heterocycles', 'FractionCSP3',
    'AmideBonds', 'LabuteASA',
    'C_Count', 'N_Count', 'O_Count', 'S_Count',
    'F_Count', 'Cl_Count', 'Br_Count', 'I_Count',
    'LipinskiRo5Violations',
]

ALERT_COLUMNS = ['PAINS', 'PAINS_Details', 'Brenk', 'Brenk_Details']
COLUMNS = (
    ['SourceLine', 'Name', 'SMILES', 'CanonicalSMILES', 'Status', 'Notes']
    + PROPERTY_COLUMNS + ALERT_COLUMNS
)


def calculate_properties(mol):
    """2D molecular descriptors, without cutoffs or drug-efficacy claims."""
    counts = Counter(atom.GetSymbol() for atom in mol.GetAtoms())
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)

    props = {
        'MW': mw,
        'ExactMW': Descriptors.ExactMolWt(mol),
        'TPSA': Descriptors.TPSA(mol),
        'cLogP': logp,
        'MolMR': Descriptors.MolMR(mol),
        'RotatableBonds': Lipinski.NumRotatableBonds(mol),
        'HBD': hbd,
        'HBA': hba,
        'FormalCharge': Chem.GetFormalCharge(mol),
        'HeavyAtomCount': Descriptors.HeavyAtomCount(mol),
        'HeteroatomCount': Descriptors.NumHeteroatoms(mol),
        'RingCount': Descriptors.RingCount(mol),
        'AromaticRings': Descriptors.NumAromaticRings(mol),
        'AliphaticRings': Descriptors.NumAliphaticRings(mol),
        'SaturatedRings': Descriptors.NumSaturatedRings(mol),
        'Heterocycles': Descriptors.NumHeterocycles(mol),
        'FractionCSP3': Descriptors.FractionCSP3(mol),
        'AmideBonds': Descriptors.NumAmideBonds(mol),
        'LabuteASA': Descriptors.LabuteASA(mol),
        'C_Count': counts['C'],
        'N_Count': counts['N'],
        'O_Count': counts['O'],
        'S_Count': counts['S'],
        'F_Count': counts['F'],
        'Cl_Count': counts['Cl'],
        'Br_Count': counts['Br'],
        'I_Count': counts['I'],
        # Informational only: the Rule of Five mainly concerns oral drugs.
        'LipinskiRo5Violations': sum((mw > 500, logp > 5, hbd > 5, hba > 10)),
    }
    try:
        props['QED'] = QED.qed(mol)
    except Exception:
        props['QED'] = None  # Missing QED does not exclude the molecule.
    return props


def catalog_alerts(mol, catalog):
    if catalog is None:
        return '', ''
    descriptions = sorted({match.GetDescription() for match in catalog.GetMatches(mol)})
    return int(bool(descriptions)), '; '.join(descriptions)



In [ ]:
with stage_run("03_profiling", [input_filename], {"structural_alerts": CALCULATE_STRUCTURAL_ALERTS}) as output_dir:
    output_smi = output_dir / "profiled_ligands.smi"
    output_csv = output_dir / "molecular_profiles.csv"
    stats = Counter()
    logger.info('Calculating descriptors: {}', input_filename)


    with open(input_filename, "r", encoding="utf-8") as f:
        total_lines = sum(1 for _ in f)

    
    with (
        input_filename.open('r', encoding='utf-8') as f_in,
        output_csv.open('w', encoding='utf-8', newline='') as f_csv,
        output_smi.open('w', encoding='utf-8') as f_smi,
    ):
        writer = csv.DictWriter(f_csv, fieldnames=COLUMNS)
        writer.writeheader()

        for line_number, line in enumerate(tqdm(f_in, unit="mol", desc='Profiling molecules', total=total_lines,
    ), start=1):
            stripped = line.strip()
            if not stripped or stripped.startswith('#'):
                stats['Skipped comments/empty'] += 1
                continue

            stats['Total'] += 1
            parts = stripped.split(maxsplit=1)
            smiles = parts[0]
            name = parts[1] if len(parts) > 1 else f'MOL_{line_number:08d}'
            row = {
                'SourceLine': line_number,
                'Name': name,
                'SMILES': smiles,
                'Status': 'OK',
                'Notes': '' if len(parts) > 1 else 'Generated name (missing in input)',
            }

            try:
                mol = Chem.MolFromSmiles(smiles)
            except Exception as exc:
                mol = None
                logger.debug('SMILES parser error on line {}: {}', line_number, exc)

            if mol is None:
                row['Status'] = 'INVALID_SMILES'
                row['Notes'] = 'RDKit could not parse/sanitize this SMILES'
                writer.writerow(row)
                stats['Invalid'] += 1
                continue

            row['CanonicalSMILES'] = Chem.MolToSmiles(mol, isomericSmiles=True)
            stats['Valid structures'] += 1
            f_smi.write(f"{row['CanonicalSMILES']} {name}\n")

            notes = [row['Notes']] if row['Notes'] else []
            try:
                row.update(calculate_properties(mol))
                if row.get('QED') is None:
                    notes.append('QED calculation unavailable')
                    stats['Missing QED'] += 1
            except Exception as exc:
                row['Status'] = 'PARTIAL'
                notes.append('Descriptor calculation error')
                logger.warning('Descriptor error on line {}: {}', line_number, exc)
                stats['Descriptor errors'] += 1

            if CALCULATE_STRUCTURAL_ALERTS:
                try:
                    row['PAINS'], row['PAINS_Details'] = catalog_alerts(mol, pains_catalog)
                    row['Brenk'], row['Brenk_Details'] = catalog_alerts(mol, brenk_catalog)
                    stats['PAINS flagged'] += row['PAINS']
                    stats['Brenk flagged'] += row['Brenk']
                except Exception as exc:
                    notes.append('Structural-alert calculation error')
                    logger.warning('Alert error on line {}: {}', line_number, exc)
                    stats['Alert errors'] += 1

            row['Notes'] = '; '.join(filter(None, notes))
            writer.writerow(row)

    logger.success('Profiling completed')
    logger.info('Valid molecules (.smi): {}', output_smi)
    logger.info('All entries + descriptors (.csv): {}', output_csv)
    logger.info('Total: {} | Valid: {} | Invalid: {}',
                stats['Total'], stats['Valid structures'], stats['Invalid'])
    logger.info('PAINS alerts: {} | Brenk alerts: {}',
                stats['PAINS flagged'], stats['Brenk flagged'])
    logger.info('Descriptor errors: {} | Alert errors: {}',
                stats['Descriptor errors'], stats['Alert errors'])
